In [ ]:
import dynamiqs as dq
import jax.numpy as jnp
import jax.scipy as jsp     # for jv via jax
from jax import device_put, numpy as jxn
import jax

# --- constants (computed once) ---
ħ = 1
e = 1
phi_quantum = 1/2

w_a = 3.7 * 2 * jnp.pi
w_b_0 = 7.05 * 2 * jnp.pi
Lamb_shift = -0.05
w_b = w_b_0

phi_a = 0.11
Z_a = 2 * phi_quantum**2 * phi_a**2
C_a_inv = Z_a * w_a
E_Ca = C_a_inv / 2

phi_b = 0.2
Z_b = 2 * phi_quantum**2 * phi_b**2
C_b_inv = Z_b * w_b
E_Cb = C_b_inv / 2

E_J = 37 * 2 * jnp.pi
kappa = 0.1 * 2 * jnp.pi

# drive/pump base params moved out
epsilon_0_base = 0.5 * jnp.pi  # choose one of your epsilon_0 values
# compute g2 using jax's bessel
g2_base = jsp.special.jv(1, epsilon_0_base) * E_J * phi_a**2 * phi_b
epsilon_d_base = 10 * g2_base
kappa_2_base = 4 * g2_base**2 / kappa

alpha = jnp.sqrt(epsilon_d_base / (2 * g2_base))

# time discretization: reduce points if possible
T = 45 / kappa
n_time = 2000   # 10000 -> 2000 for speed; increase if needed
tsave = jnp.linspace(0, T, n_time)

# prepare n_a range and preallocate outputs
n_a_values = list(range(15, 25))
N = len(n_a_values)
delta_a_max = jnp.zeros(N)
delta_b_max = jnp.zeros(N)



@jax.jit
def run_single(n_a):
    n_b = (n_a + 1) // 2
    a = dq.destroy(n_a)
    b = dq.destroy(n_b)

    varphi_a_op_tot = dq.tensor(jnp.sqrt(Z_a/2)*(a + a.dag()) / phi_quantum, dq.eye(n_b))
    varphi_b_op_tot = dq.tensor(dq.eye(n_a), jnp.sqrt(Z_b/2)*(b + b.dag()) / phi_quantum)

    psi_0 = dq.tensor(dq.fock(n_a, 1), dq.fock(n_b, 0))

    drive = lambda t: epsilon_d * jnp.cos(w_d * t) 
    H_drive = dq.modulated(drive, dq.tensor(dq.eye(n_a), b + b.dag()))

    pump = lambda t: - 2 * E_J * jnp.sin(epsilon_0 * jnp.cos(w_p * t))
    H_pump = dq.modulated(pump, dq.sinm(varphi_a_op_tot + varphi_b_op_tot) - varphi_a_op_tot - varphi_b_op_tot)

    H = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b)) \
      + dq.tensor(dq.eye(n_a), w_b_0 * dq.number(n_b)) \
      + H_drive + H_pump

    jumps = [jnp.sqrt(2*kappa) * dq.tensor(dq.eye(n_a), b)]

    result = dq.mesolve(H, jumps, psi_0, tsave, method=dq.method.Tsit5(max_steps=20000000))

    aadag_mean = dq.expect(dq.tensor(a@a.dag(), dq.eye(n_b)), result.states)
    adaga_mean = dq.expect(dq.tensor(a.dag()@a, dq.eye(n_b)), result.states)
    bbdag_mean = dq.expect(dq.tensor(dq.eye(n_a), b@b.dag()), result.states)
    bdagb_mean = dq.expect(dq.tensor(dq.eye(n_a), b.dag()@b), result.states)

    delta_a = jnp.max(jnp.abs(1 - (aadag_mean - adaga_mean).real))
    delta_b = jnp.max(jnp.abs(1 - (bbdag_mean - bdagb_mean).real))

    return delta_a, delta_b


for idx, n_a in enumerate(n_a_values):
    da, db = run_single(n_a)
    delta_a_max[idx] = da
    delta_b_max[idx] = db